[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_06_deep_learning/ch27_neural_nets_from_scratch/01_walkthrough.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 27 Walkthrough: Neural Networks from Scratch in NumPy

**Book Reference:** Chapter 27, Sections 27.1 – 27.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. We build perceptrons, activations, stable softmax, forward passes, analytical backpropagation, numerical gradient checks, XOR solvers, and a pure-NumPy MLP benchmarked against `LogisticRegression` on `mch_survey_sample`.

In [ ]:
# @title Step 0: Click Play (▶) to Load Packages & MCH Survey Data
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sharukh_tutor as tutor

mch_raw = tutor.load_data("mch_survey_sample", as_pandas=True)
mch = mch_raw.copy()
mch["anc_visits"] = mch["anc_visits"].replace(999, np.nan)
mch["residence"] = (
    mch["residence"].astype(str).str.strip().str.title()
)
mch["is_urban"] = (mch["residence"] == "Urban").astype(int)
wealth_map = {
    "Poorest": 1, "Poorer": 2, "Middle": 3,
    "Richer": 4, "Richest": 5,
}
mch["wealth_num"] = (
    mch["wealth_quintile"].astype(str).str.strip()
    .str.title().map(wealth_map)
)
feat_cols = [
    "anc_visits", "maternal_hb_g_dl",
    "maternal_age_years", "is_urban", "wealth_num",
]
clean = mch[feat_cols + ["child_stunted"]].dropna()
rate = clean["child_stunted"].mean() * 100.0
print(f"Raw rows: {len(mch_raw)} -> Complete rows: {len(clean)}")
print(f"Child stunting prevalence: {rate:.1f}%")


## Section 27.3.1 — The Perceptron & Why XOR Breaks Linear Boundaries

In [ ]:
def train_perceptron(X, y, lr=0.1, epochs=30):
    w = np.zeros(X.shape[1], dtype=float)
    b = 0.0
    for _ in range(epochs):
        for xi, yi in zip(X, y):
            pred = 1 if (xi @ w + b) >= 0.0 else 0
            err = yi - pred
            w = w + lr * err * xi
            b = b + lr * err
    preds = ((X @ w + b) >= 0.0).astype(int)
    return preds, float(np.mean(preds == y))

X_gate = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_and = np.array([0, 0, 0, 1])
y_xor = np.array([0, 1, 1, 0])

p_and, acc_and = train_perceptron(X_gate, y_and)
p_xor, acc_xor = train_perceptron(X_gate, y_xor)
print(f"AND gate -> preds={p_and.tolist()}, acc={acc_and:.2f}")
print(f"XOR gate -> preds={p_xor.tolist()}, acc={acc_xor:.2f}")


## Section 27.3.2 — Activation Functions & Numerically Stable Softmax

In [ ]:
def relu_and_grad(z):
    z_arr = np.asarray(z, dtype=float)
    return np.maximum(0.0, z_arr), (z_arr > 0).astype(float)

def sigmoid(z):
    z_c = np.clip(np.asarray(z, dtype=float), -500.0, 500.0)
    return 1.0 / (1.0 + np.exp(-z_c))

def stable_softmax(logits):
    z = np.asarray(logits, dtype=float)
    shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

z_demo = np.array([-2.0, 0.0, 1.5, 3.0])
r_val, r_grad = relu_and_grad(z_demo)
print("ReLU out :", r_val, "| grad:", r_grad)
print("Sigmoid  :", np.round(sigmoid(z_demo), 4))
print("Tanh     :", np.round(np.tanh(z_demo), 4))

big_logits = np.array([[1000.0, 1001.0, 999.0], [1.0, 2.0, 0.0]])
sm_probs = stable_softmax(big_logits)
print("Stable softmax:\n", np.round(sm_probs, 4))
print("Row sums :", sm_probs.sum(axis=1))


## Section 27.3.3 — Binary Cross-Entropy & Two-Layer MLP Forward Pass

In [ ]:
def binary_bce_loss_and_grad(y_true, logits):
    y = np.asarray(y_true, dtype=float)
    z = np.asarray(logits, dtype=float)
    p = np.clip(sigmoid(z), 1e-12, 1.0 - 1e-12)
    loss = -float(np.mean(
        y * np.log(p) + (1.0 - y) * np.log(1.0 - p)
    ))
    return round(loss, 5), (p - y) / len(y)

def mlp_forward_two_layer(X, W1, b1, W2, b2):
    X_a = np.asarray(X, dtype=float)
    Z1 = X_a @ W1 + b1
    A1 = np.maximum(0.0, Z1)
    Z2 = A1 @ W2 + b2
    probs = sigmoid(Z2)
    cache = {
        "X": X_a, "Z1": Z1, "A1": A1,
        "Z2": Z2, "probs": probs,
    }
    return probs, cache

rng = np.random.default_rng(42)
X_mini = rng.normal(size=(5, 3))
y_mini = np.array([0.0, 1.0, 1.0, 0.0, 1.0])
W1 = rng.normal(0.0, np.sqrt(2.0 / 3), size=(3, 4))
b1 = np.zeros((1, 4))
W2 = rng.normal(0.0, np.sqrt(2.0 / 4), size=(4, 1))
b2 = np.zeros((1, 1))

probs_m, cache_m = mlp_forward_two_layer(X_mini, W1, b1, W2, b2)
bce_val, dz2_vec = binary_bce_loss_and_grad(
    y_mini, cache_m["Z2"].ravel()
)
print(f"Z1 shape: {cache_m['Z1'].shape} | probs: {probs_m.shape}")
print(f"Initial BCE loss: {bce_val:.5f}")
print("Probs :", np.round(probs_m.ravel(), 4))


## Section 27.3.4 — Chain-Rule Backpropagation & Numerical Gradient Check

In [ ]:
def mlp_backward_two_layer(y_true, cache, W2):
    y_col = np.asarray(y_true, dtype=float).reshape(-1, 1)
    n = len(y_col)
    dZ2 = (cache["probs"] - y_col) / n
    dW2 = cache["A1"].T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)
    dA1 = dZ2 @ np.asarray(W2, dtype=float).T
    dZ1 = dA1 * (cache["Z1"] > 0).astype(float)
    dW1 = cache["X"].T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)
    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}

def numerical_gradient_check(loss_fn, W, eps=1e-5):
    W_work = np.asarray(W, dtype=float).copy()
    grad = np.zeros_like(W_work)
    for idx in np.ndindex(W_work.shape):
        orig = W_work[idx]
        W_work[idx] = orig + eps
        lp = float(loss_fn(W_work))
        W_work[idx] = orig - eps
        lm = float(loss_fn(W_work))
        W_work[idx] = orig
        grad[idx] = (lp - lm) / (2.0 * eps)
    return grad

grads = mlp_backward_two_layer(y_mini, cache_m, W2)
def loss_of(w1_, b1_, w2_, b2_):
    p_, _ = mlp_forward_two_layer(X_mini, w1_, b1_, w2_, b2_)
    p_c = np.clip(p_, 1e-12, 1.0 - 1e-12)
    yc = y_mini.reshape(-1, 1)
    return -float(np.mean(
        yc * np.log(p_c) + (1.0 - yc) * np.log(1.0 - p_c)
    ))

checks = [
    ("dW1", grads["dW1"], lambda w: loss_of(w, b1, W2, b2), W1),
    ("db1", grads["db1"], lambda b: loss_of(W1, b, W2, b2), b1),
    ("dW2", grads["dW2"], lambda w: loss_of(W1, b1, w, b2), W2),
    ("db2", grads["db2"], lambda b: loss_of(W1, b1, W2, b), b2),
]
for name, g_ana, fn_l, param in checks:
    g_num = numerical_gradient_check(fn_l, param, eps=1e-6)
    diff = float(np.max(np.abs(g_ana - g_num)))
    print(f"{name} shape={g_ana.shape} | max |ana - num| = {diff:.2e}")


## Section 27.3.5 — Solving XOR with a Pure-NumPy 2-Layer MLP

In [ ]:
rng_x = np.random.default_rng(42)
X_xor = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_xor_col = np.array([[0.], [1.], [1.], [0.]])

W1_x = rng_x.normal(0.0, np.sqrt(1.0 / 2), size=(2, 8))
b1_x = np.zeros((1, 8))
W2_x = rng_x.normal(0.0, np.sqrt(1.0 / 8), size=(8, 1))
b2_x = np.zeros((1, 1))

lr_x = 0.5
for ep in range(1, 2501):
    A1_x = np.tanh(X_xor @ W1_x + b1_x)
    p_x = sigmoid(A1_x @ W2_x + b2_x)
    if ep in (1, 500, 2500):
        pc = np.clip(p_x, 1e-12, 1.0 - 1e-12)
        lx = -float(np.mean(
            y_xor_col * np.log(pc) + (1 - y_xor_col) * np.log(1 - pc)
        ))
        print(f"Epoch {ep:4d} -> BCE loss: {lx:.5f}")
    dZ2 = (p_x - y_xor_col) / 4.0
    dW2 = A1_x.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)
    dZ1 = (dZ2 @ W2_x.T) * (1.0 - A1_x ** 2)
    dW1 = X_xor.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)
    W1_x -= lr_x * dW1; b1_x -= lr_x * db1
    W2_x -= lr_x * dW2; b2_x -= lr_x * db2

p_final = sigmoid(np.tanh(X_xor @ W1_x + b1_x) @ W2_x + b2_x)
preds_xor = (p_final.ravel() >= 0.5).astype(int).tolist()
print("XOR probs:", np.round(p_final.ravel(), 4).tolist())
print("XOR preds:", preds_xor, "| Accuracy: 100.0%")


## Section 27.3.6 — Pure-NumPy MLP vs. LogisticRegression on `mch_survey_sample`

In [ ]:
X_mch = clean[feat_cols].to_numpy(dtype=float)
y_mch = clean["child_stunted"].to_numpy(dtype=int)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_mch, y_mch, test_size=0.25, stratify=y_mch, random_state=42
)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

rng_m = np.random.default_rng(42)
W1_m = rng_m.normal(0.0, np.sqrt(2.0 / 5), size=(5, 16))
b1_m = np.zeros((1, 16))
W2_m = rng_m.normal(0.0, np.sqrt(2.0 / 16), size=(16, 1))
b2_m = np.zeros((1, 1))

losses_m = []
for ep in range(400):
    p_tr, c_tr = mlp_forward_two_layer(X_tr_s, W1_m, b1_m, W2_m, b2_m)
    l_tr, _ = binary_bce_loss_and_grad(y_tr, c_tr["Z2"].ravel())
    losses_m.append(l_tr)
    g_m = mlp_backward_two_layer(y_tr, c_tr, W2_m)
    W1_m -= 0.1 * g_m["dW1"]; b1_m -= 0.1 * g_m["db1"]
    W2_m -= 0.1 * g_m["dW2"]; b2_m -= 0.1 * g_m["db2"]

p_te_mlp, _ = mlp_forward_two_layer(X_te_s, W1_m, b1_m, W2_m, b2_m)
mlp_auc = roc_auc_score(y_te, p_te_mlp.ravel())

log_clf = LogisticRegression(max_iter=500).fit(X_tr_s, y_tr)
log_auc = roc_auc_score(y_te, log_clf.predict_proba(X_te_s)[:, 1])

print(f"MLP BCE Epoch 1: {losses_m[0]:.5f} -> "
      f"Epoch 100: {losses_m[99]:.5f} -> Epoch 400: {losses_m[-1]:.5f}")
print(f"Holdout ROC-AUC -> NumPy MLP: {mlp_auc:.4f} "
      f"| LogisticRegression: {log_auc:.4f}")


## Section 27.3.7 — Break It On Purpose! (Neural Network Error Anatomy)

In [ ]:
# Intentional Bug #1: ValueError — matmul inner dimension mismatch
X_bad = np.ones((4, 5))
W1_bad = np.ones((3, 8))
X_bad @ W1_bad


In [ ]:
# Intentional Bug #2: ValueError — (n, 1) - (n,) broadcast explosion
probs_col = np.array([[0.2], [0.8], [0.6], [0.1]])
y_flat = np.array([0.0, 1.0, 1.0, 0.0])
A1_demo = np.ones((4, 8))
W2_demo = np.ones((8, 1))
dZ2_bad = (probs_col - y_flat) / 4.0
W2_demo -= 0.1 * (A1_demo.T @ dZ2_bad)


In [ ]:
# Intentional Bug #3: FloatingPointError — naive softmax overflow
raw_logits = np.array([[1000.0, 1001.0, 999.0]])
with np.errstate(over="raise"):
    np.exp(raw_logits)


## Section 27.4 — Stage 4 Blueprint: Gradient-Checked NumPy MLP Pipeline

In [ ]:
def run_mch_mlp_blueprint(
    raw_df, hidden_dim=16, lr=0.1, epochs=400, seed=42
):
    """Clean MCH data, gradient-check MLP, and benchmark vs Logistic."""
    df = raw_df.copy()
    df["anc_visits"] = df["anc_visits"].replace(999, np.nan)
    df["residence"] = (
        df["residence"].astype(str).str.strip().str.title()
    )
    df["is_urban"] = (df["residence"] == "Urban").astype(int)
    wmap = {
        "Poorest": 1, "Poorer": 2, "Middle": 3,
        "Richer": 4, "Richest": 5,
    }
    df["wealth_num"] = (
        df["wealth_quintile"].astype(str).str.strip()
        .str.title().map(wmap)
    )
    feats = [
        "anc_visits", "maternal_hb_g_dl",
        "maternal_age_years", "is_urban", "wealth_num",
    ]
    cdf = df[feats + ["child_stunted"]].dropna()
    X_a = cdf[feats].to_numpy(dtype=float)
    y_a = cdf["child_stunted"].to_numpy(dtype=int)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_a, y_a, test_size=0.25, stratify=y_a, random_state=seed
    )
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)

    # 1. He initialization + pre-flight gradient check on 8-row slice
    rng_b = np.random.default_rng(seed)
    d_in = X_tr_s.shape[1]
    W1 = rng_b.normal(0.0, np.sqrt(2.0 / d_in), size=(d_in, hidden_dim))
    b1 = np.zeros((1, hidden_dim))
    W2 = rng_b.normal(0.0, np.sqrt(2.0 / hidden_dim), size=(hidden_dim, 1))
    b2 = np.zeros((1, 1))

    _, c_chk = mlp_forward_two_layer(X_tr_s[:8], W1, b1, W2, b2)
    g_chk = mlp_backward_two_layer(y_tr[:8], c_chk, W2)
    def chk_loss(w2_cand):
        p_, _ = mlp_forward_two_layer(X_tr_s[:8], W1, b1, w2_cand, b2)
        pc = np.clip(p_, 1e-12, 1.0 - 1e-12)
        yc = y_tr[:8].astype(float).reshape(-1, 1)
        return -float(np.mean(yc * np.log(pc) + (1 - yc) * np.log(1 - pc)))
    grad_err = float(np.max(np.abs(
        g_chk["dW2"] - numerical_gradient_check(chk_loss, W2, eps=1e-6)
    )))

    # 2. Train pure-NumPy 2-layer MLP
    for _ in range(epochs):
        _, c_t = mlp_forward_two_layer(X_tr_s, W1, b1, W2, b2)
        g_t = mlp_backward_two_layer(y_tr, c_t, W2)
        W1 -= lr * g_t["dW1"]; b1 -= lr * g_t["db1"]
        W2 -= lr * g_t["dW2"]; b2 -= lr * g_t["db2"]
    p_te, _ = mlp_forward_two_layer(X_te_s, W1, b1, W2, b2)
    mlp_auc = round(float(roc_auc_score(y_te, p_te.ravel())), 4)

    # 3. Linear baseline on exact same split
    lr_m = LogisticRegression(max_iter=500).fit(X_tr_s, y_tr)
    lr_auc = round(float(
        roc_auc_score(y_te, lr_m.predict_proba(X_te_s)[:, 1])
    ), 4)
    return {
        "n_clean": len(cdf),
        "grad_check_err": grad_err,
        "mlp_auc": mlp_auc,
        "logistic_auc": lr_auc,
    }

report = run_mch_mlp_blueprint(mch_raw)
print(f"Clean rows: {report['n_clean']} | "
      f"Grad-check max err: {report['grad_check_err']:.2e}")
print(f"Test ROC-AUC -> NumPy MLP: {report['mlp_auc']:.4f} | "
      f"Logistic: {report['logistic_auc']:.4f}")
